In [1]:
import pandas as pd
import psycopg

IMPULSE_PCT_ATR = 0.15

RTH_START = "08:30"
RTH_END = "14:55"

In [2]:
query = """
SELECT
    timestamp,
    open,
    high,
    low,
    close,
    volume
FROM candles
ORDER BY timestamp
"""

with psycopg.connect("dbname=dailyedge_development") as connection:
    df = pd.read_sql(query, connection)

df["Datetime"] = pd.to_datetime(df["timestamp"])
df = df.drop(columns=["timestamp"])

print(f"{len(df):,} rows loaded")
df.head()

/tmp/ipykernel_3680/2604799363.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


5,884,751 rows loaded


,open,high,low,close,volume,Datetime
0,1416.299899,1416.299899,1416.299899,1416.299899,1,2008-12-11 01:38:00
1,1412.231742,1412.231742,1412.231742,1412.231742,1,2008-12-11 01:52:00
2,1410.197663,1410.197663,1408.744750,1408.744750,5,2008-12-11 02:14:00
3,1408.454167,1408.454167,1408.454167,1408.454167,1,2008-12-11 02:15:00
4,1407.291836,1407.291836,1407.291836,1407.291836,6,2008-12-11 02:17:00


In [3]:
df["Date"] = df["Datetime"].dt.date

df["SessionDate"] = (
    df["Datetime"]
    + pd.to_timedelta((df["Datetime"].dt.hour >= 17).astype(int), unit="D")
).dt.date


def compute_daily_atr(df):
    daily = (
        df.groupby("SessionDate")
        .agg(
            Open=("open", "first"),
            High=("high", "max"),
            Low=("low", "min"),
            Close=("close", "last"),
        )
    )

    daily["PrevClose"] = daily["Close"].shift(1)

    daily["TR"] = pd.concat(
        [
            daily["High"] - daily["Low"],
            (daily["High"] - daily["PrevClose"]).abs(),
            (daily["Low"] - daily["PrevClose"]).abs(),
        ],
        axis=1,
    ).max(axis=1)

    daily["ATR14"] = daily["TR"].rolling(14).mean().shift(1)

    return daily


daily = compute_daily_atr(df)

df = df.merge(
    daily[["ATR14"]],
    left_on="SessionDate",
    right_index=True,
    how="left",
)

df.head()

,open,high,low,close,volume,Datetime,Date,SessionDate,ATR14
0,1416.299899,1416.299899,1416.299899,1416.299899,1,2008-12-11 01:38:00,2008-12-11,2008-12-11,NaN
1,1412.231742,1412.231742,1412.231742,1412.231742,1,2008-12-11 01:52:00,2008-12-11,2008-12-11,NaN
2,1410.197663,1410.197663,1408.744750,1408.744750,5,2008-12-11 02:14:00,2008-12-11,2008-12-11,NaN
3,1408.454167,1408.454167,1408.454167,1408.454167,1,2008-12-11 02:15:00,2008-12-11,2008-12-11,NaN
4,1407.291836,1407.291836,1407.291836,1407.291836,6,2008-12-11 02:17:00,2008-12-11,2008-12-11,NaN


In [4]:
df = df[df["Datetime"] >= "2022-01-01"].copy()

print(f"{len(df):,} rows after filtering")

1,590,256 rows after filtering


In [5]:
def first_impulse(rth):
    opening_price = rth.iloc[0]["open"]
    impulse = rth.iloc[0]["ATR14"] * IMPULSE_PCT_ATR

    up_target = opening_price + impulse
    down_target = opening_price - impulse

    for index, (_, candle) in enumerate(rth.iterrows()):
        hit_up = candle["high"] >= up_target
        hit_down = candle["low"] <= down_target

        if hit_up and hit_down:
            return "Unknown", index

        if hit_up:
            return "Up", index

        if hit_down:
            return "Down", index

    return "Neither", None

In [6]:
def evaluate_after_impulse(rth, direction, impulse_idx, atr14):
    n_atr = atr14 * IMPULSE_PCT_ATR
    opening_price = rth.iloc[0]["open"]

    if direction == "Up":
        impulse_price = opening_price + n_atr
        continuation_price = impulse_price + (2 * n_atr)
        retracement_price = impulse_price - n_atr

    elif direction == "Down":
        impulse_price = opening_price - n_atr
        continuation_price = impulse_price - (2 * n_atr)
        retracement_price = impulse_price + n_atr

    else:
        return None

    forward = rth.iloc[impulse_idx + 1:]

    for _, candle in forward.iterrows():
        if direction == "Up":
            continuation_hit = candle["high"] >= continuation_price
            retracement_hit = candle["low"] <= retracement_price
        else:
            continuation_hit = candle["low"] <= continuation_price
            retracement_hit = candle["high"] >= retracement_price

        if continuation_hit and retracement_hit:
            return {
                "Outcome": "Ambiguous",
                "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                "ContinuationTime": candle["Datetime"].strftime("%H:%M"),
                "RetracementTime": candle["Datetime"].strftime("%H:%M"),
                "ImpulsePrice": impulse_price,
                "ContinuationPrice": continuation_price,
                "RetracementPrice": retracement_price,
            }

        if continuation_hit:
            return {
                "Outcome": "Continuation",
                "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                "ContinuationTime": candle["Datetime"].strftime("%H:%M"),
                "RetracementTime": "-",
                "ImpulsePrice": impulse_price,
                "ContinuationPrice": continuation_price,
                "RetracementPrice": retracement_price,
            }

        if retracement_hit:
            return {
                "Outcome": "Retracement",
                "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                "ContinuationTime": "-",
                "RetracementTime": candle["Datetime"].strftime("%H:%M"),
                "ImpulsePrice": impulse_price,
                "ContinuationPrice": continuation_price,
                "RetracementPrice": retracement_price,
            }

    return {
        "Outcome": "Neither",
        "OutcomeTime": "-",
        "ContinuationTime": "-",
        "RetracementTime": "-",
        "ImpulsePrice": impulse_price,
        "ContinuationPrice": continuation_price,
        "RetracementPrice": retracement_price,
    }

In [ ]:
results = []

for date, session in df.groupby("Date"):

    session_times = session["Datetime"].dt.strftime("%H:%M")

    rth = session[
        (session_times >= RTH_START)
        & (session_times <= RTH_END)
    ]

    if rth.empty:
        continue

    if pd.isna(rth.iloc[0]["ATR14"]):
        continue

    direction, impulse_idx = first_impulse(rth)

    if direction not in ("Up", "Down"):
        continue

    outcome = evaluate_after_impulse(
        rth,
        direction,
        impulse_idx,
        rth.iloc[0]["ATR14"],
    )

    results.append({
        "Date": date,
        "Direction": direction,
        **outcome,
    })

results = pd.DataFrame(results)

print(results["Outcome"].value_counts())
display(results.head())

In [ ]:
counts = results["Outcome"].value_counts()

total = counts["Continuation"] + counts["Retracement"]

summary = (
    counts.loc[["Continuation", "Retracement"]]
    .rename("Count")
    .to_frame()
)

summary["Percent"] = (summary["Count"] / total * 100).round(2)

display(summary)

print(f"\nTotal evaluated sessions: {total}")

,Count,Percent
Outcome,,
Continuation,381,35.44
Retracement,694,64.56



Total evaluated sessions: 1075


In [ ]:
# ==========================================================
# Experiment: TP = 1× Impulse | SL = 2× Impulse
# ==========================================================

EXPERIMENT_NAME = "TP = 1× Impulse | SL = 2× Impulse"

RTH_START = "08:30"
RTH_END = "14:55"

sessions = df.groupby(df["Datetime"].dt.date)


def evaluate_after_impulse_tp1_sl2(rth, direction, impulse_idx):
    impulse = rth.iloc[0]["ATR14"] * IMPULSE_PCT_ATR

    impulse_price = (
        rth.iloc[0]["open"] + impulse
        if direction == "Up"
        else rth.iloc[0]["open"] - impulse
    )

    if direction == "Up":
        continuation_target = impulse_price + impulse
        retracement_target = impulse_price - 2 * impulse
    else:
        continuation_target = impulse_price - impulse
        retracement_target = impulse_price + 2 * impulse

    forward = rth.iloc[impulse_idx + 1 :]

    for _, candle in forward.iterrows():

        if direction == "Up":

            if candle["high"] >= continuation_target:
                return {
                    "Outcome": "Continuation",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": candle["Datetime"].strftime("%H:%M"),
                    "RetracementTime": "-",
                }

            if candle["low"] <= retracement_target:
                return {
                    "Outcome": "Retracement",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": "-",
                    "RetracementTime": candle["Datetime"].strftime("%H:%M"),
                }

        else:

            if candle["low"] <= continuation_target:
                return {
                    "Outcome": "Continuation",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": candle["Datetime"].strftime("%H:%M"),
                    "RetracementTime": "-",
                }

            if candle["high"] >= retracement_target:
                return {
                    "Outcome": "Retracement",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": "-",
                    "RetracementTime": candle["Datetime"].strftime("%H:%M"),
                }

    return {
        "Outcome": "Neither",
        "OutcomeTime": "-",
        "ContinuationTime": "-",
        "RetracementTime": "-",
    }


results_tp1_sl2 = []

for date, session in sessions:

    session_times = session["Datetime"].dt.strftime("%H:%M")
    rth = session[(session_times >= RTH_START) & (session_times <= RTH_END)]

    if rth.empty or pd.isna(rth.iloc[0]["ATR14"]):
        continue

    direction, impulse_idx = first_impulse(rth)

    if direction in ("Unknown", "Neither") or impulse_idx is None:
        continue

    result = evaluate_after_impulse_tp1_sl2(
        rth,
        direction,
        impulse_idx,
    )

    results_tp1_sl2.append(
        {
            "Date": date,
            "Direction": direction,
            **result,
        }
    )

results_tp1_sl2 = pd.DataFrame(results_tp1_sl2)

counts = results_tp1_sl2["Outcome"].value_counts()

total = counts.get("Continuation", 0) + counts.get("Retracement", 0)

summary = (
    counts.reindex(["Continuation", "Retracement"], fill_value=0)
    .rename("Count")
    .to_frame()
)

summary["Percent"] = (summary["Count"] / total * 100).round(2)

wins = counts.get("Continuation", 0)
losses = counts.get("Retracement", 0)

net_r = wins - 2 * losses

print("=" * 60)
print(EXPERIMENT_NAME)
print("=" * 60)

display(summary)

print(f"\nWins: {wins}")
print(f"Losses: {losses}")
print(f"Net Result: {net_r}R")
print(f"Total evaluated sessions: {total}")

TP = 1× Impulse | SL = 2× Impulse


,Count,Percent
Outcome,,
Continuation,759,69.7
Retracement,330,30.3



Wins: 759
Losses: 330
Net Result: 99R
Total evaluated sessions: 1089


In [ ]:
# ==========================================================
# Experiment: TP = 1× Impulse | SL = 1× Impulse
# ==========================================================

EXPERIMENT_NAME = "TP = 1× Impulse | SL = 1× Impulse"

RTH_START = "08:30"
RTH_END = "14:55"

sessions = df.groupby(df["Datetime"].dt.date)


def evaluate_after_impulse_tp1_sl1(rth, direction, impulse_idx):
    impulse = rth.iloc[0]["ATR14"] * IMPULSE_PCT_ATR

    impulse_price = (
        rth.iloc[0]["open"] + impulse
        if direction == "Up"
        else rth.iloc[0]["open"] - impulse
    )

    if direction == "Up":
        continuation_target = impulse_price + impulse
        retracement_target = impulse_price - impulse
    else:
        continuation_target = impulse_price - impulse
        retracement_target = impulse_price + impulse

    forward = rth.iloc[impulse_idx + 1 :]

    for _, candle in forward.iterrows():

        if direction == "Up":

            if candle["high"] >= continuation_target:
                return {
                    "Outcome": "Continuation",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": candle["Datetime"].strftime("%H:%M"),
                    "RetracementTime": "-",
                }

            if candle["low"] <= retracement_target:
                return {
                    "Outcome": "Retracement",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": "-",
                    "RetracementTime": candle["Datetime"].strftime("%H:%M"),
                }

        else:

            if candle["low"] <= continuation_target:
                return {
                    "Outcome": "Continuation",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": candle["Datetime"].strftime("%H:%M"),
                    "RetracementTime": "-",
                }

            if candle["high"] >= retracement_target:
                return {
                    "Outcome": "Retracement",
                    "OutcomeTime": candle["Datetime"].strftime("%H:%M"),
                    "ContinuationTime": "-",
                    "RetracementTime": candle["Datetime"].strftime("%H:%M"),
                }

    return {
        "Outcome": "Neither",
        "OutcomeTime": "-",
        "ContinuationTime": "-",
        "RetracementTime": "-",
    }


results_tp1_sl1 = []

for date, session in sessions:

    session_times = session["Datetime"].dt.strftime("%H:%M")
    rth = session[(session_times >= RTH_START) & (session_times <= RTH_END)]

    if rth.empty or pd.isna(rth.iloc[0]["ATR14"]):
        continue

    direction, impulse_idx = first_impulse(rth)

    if direction in ("Unknown", "Neither") or impulse_idx is None:
        continue

    result = evaluate_after_impulse_tp1_sl1(
        rth,
        direction,
        impulse_idx,
    )

    results_tp1_sl1.append(
        {
            "Date": date,
            "Direction": direction,
            **result,
        }
    )

results_tp1_sl1 = pd.DataFrame(results_tp1_sl1)

counts = results_tp1_sl1["Outcome"].value_counts()

total = counts.get("Continuation", 0) + counts.get("Retracement", 0)

summary = (
    counts.reindex(["Continuation", "Retracement"], fill_value=0)
    .rename("Count")
    .to_frame()
)

summary["Percent"] = (summary["Count"] / total * 100).round(2)

wins = counts.get("Continuation", 0)
losses = counts.get("Retracement", 0)

net_r = wins - losses

print("=" * 60)
print(EXPERIMENT_NAME)
print("=" * 60)

display(summary)

print(f"\nWins: {wins}")
print(f"Losses: {losses}")
print(f"Net Result: {net_r}R")
print(f"Total evaluated sessions: {total}")

TP = 1× Impulse | SL = 1× Impulse


,Count,Percent
Outcome,,
Continuation,578,51.84
Retracement,537,48.16



Wins: 578
Losses: 537
Net Result: 41R
Total evaluated sessions: 1115


In [ ]:
unknown_total = 0
unknown_tp = 0

for _, session in df.groupby("SessionDate"):

    session_times = session["Datetime"].dt.strftime("%H:%M")

    rth = session[
        (session_times >= RTH_START)
        & (session_times <= RTH_END)
    ]

    if rth.empty:
        continue

    if pd.isna(rth.iloc[0]["ATR14"]):
        continue

    direction, impulse_idx = first_impulse(rth)

    if direction != "Unknown":
        continue

    unknown_total += 1

    opening = rth.iloc[0]["open"]
    impulse = rth.iloc[0]["ATR14"] * IMPULSE_PCT_ATR

    # first_impulse() currently returns None for Unknown,
    # which means the ambiguous candle is the one that triggered Unknown.
    # At the moment that's always the current candle in the loop,
    # but since the function doesn't expose its index we'll assume
    # it occurred on the opening candle.
    candle = rth.iloc[0]

    if (
        candle["high"] >= opening + 2 * impulse
        or
        candle["low"] <= opening - 2 * impulse
    ):
        unknown_tp += 1

print(f"Unknown sessions : {unknown_total}")
print(f"Reached 1R TP    : {unknown_tp}")
print(f"Percent          : {unknown_tp/unknown_total:.1%}")

Unknown sessions : 0
Reached 1R TP    : 0


ZeroDivisionError: division by zero

In [ ]:
processed_sessions = []

categories = {
    "TP only": 0,
    "SL only": 0,
    "TP and SL": 0,
    "Neither": 0,
}

for _, session in df.groupby("SessionDate"):

    session_times = session["Datetime"].dt.strftime("%H:%M")

    rth = session[
        (session_times >= RTH_START)
        & (session_times <= RTH_END)
    ]

    if rth.empty:
        continue

    if pd.isna(rth.iloc[0]["ATR14"]):
        continue

    direction, impulse_idx = first_impulse(rth)

    if direction != "Unknown":
        continue

    opening = rth.iloc[0]["open"]
    impulse = rth.iloc[0]["ATR14"] * IMPULSE_PCT_ATR

    candle = rth.iloc[impulse_idx]

    long_tp = opening + 2 * impulse
    short_tp = opening - 2 * impulse

    long_sl = opening
    short_sl = opening

    hit_tp = (
        candle["high"] >= long_tp
        or
        candle["low"] <= short_tp
    )

    hit_sl = (
        candle["low"] <= long_sl
        or
        candle["high"] >= short_sl
    )

    processed_sessions.append({
    "SessionDate": session["SessionDate"].iloc[0],
    "Direction": direction,
    })

    if hit_tp and hit_sl:
        categories["TP and SL"] += 1
    elif hit_tp:
        categories["TP only"] += 1
    elif hit_sl:
        categories["SL only"] += 1
    else:
        categories["Neither"] += 1

results = pd.DataFrame(processed_sessions)

print("Rows:", len(results))
print("Unique sessions:", results["SessionDate"].nunique())
print("Date range:",
      results["SessionDate"].min(),
      "to",
      results["SessionDate"].max())

summary = (
    pd.DataFrame.from_dict(categories, orient="index", columns=["Count"])
    .rename_axis("Category")
    .reset_index()
)

summary["Percent"] = (
    summary["Count"] / summary["Count"].sum() * 100
).round(1)

display(summary)

Rows: 0


KeyError: 'SessionDate'